# ReFuelEU optimisation — runs

Every optimisation behind the paper and its revision. There are three families, and they
are driven from a **terminal**, not from this notebook: each run is a subprocess, they
are run several at a time, and a notebook kernel would serialise them.

| family | what varies | runs | where |
|---|---|---|---|
| **blocks A–D** | one parameter at a time, at a single carbon budget | 14 | `sweep/results/` |
| **block E** | the carbon budget, for five biomass/technology cases | 55 | `sweep/results_e/` |
| **references** | nothing — fixed-mandate MDAs | 8 + 3 | `sweep/results/`, `sweep/results_e/` |

The cells below are the commands, with what each one is for. The last two sections read
results back and are the only ones that compute anything here.

## The two budgets

Blocks A–D are all held at **one** budget: the cumulative 2020–2050 CO₂ of ReFuelEU
(linear) at ε_P = −0.9, which is **3.8656495 GtCO₂** on the EU perimeter, or
**3.122636344 %** of the world aviation budget. It is measured rather than chosen —
`preflight.py` computes it from the reference run and says so if it has moved.

It moved once, from 3.8615905850 GtCO₂: the mandate used to phase biofuel in linearly
from 2020, and the phantom 2021–2024 biofuel abated 4.06 MtCO₂ inside the very run that
defines the ceiling. ReFuelEU's first obligation is a step at 2025, and so is the model's
now.

The constraint takes a *share*, but a share is an absolute budget here: when it is active
it pins `cumulative_co2_emissions[2050] = gross_carbon_budget_2050 × share / 100`, and
both factors on the right are exogenous.

Block E sweeps the published ladder instead — 3.8 down to 2.0 % of the world budget, plus
min-CO₂ — deliberately not re-centred on the ReFuelEU value, since the point of that
surface is to explore ambitious scenarios. 3.123 % is *marked* on it, between the 3.2 and
3.0 rungs.

## The five cases

| `CASE` | biomass to aviation | efficiency gain | published as |
|---|---|---|---|
| `main` | **10 %** | 1.35 %/yr | reference case |
| `B5` | 5 % | 1.35 %/yr | biomass sensitivity |
| `B75` | 7.5 % | 1.35 %/yr | biomass sensitivity |
| `B15` | 15 % | 1.35 %/yr | biomass sensitivity |
| `pess` | **10 %** | 0.91 %/yr | pessimistic technology roadmap |

`main` and `pess` carried 9.90 % in the submitted version. That value was reverse-
engineered so that production efficiency covered 2019 aviation energy use, which makes a
convention look derived; a round 10 % is equally arbitrary and visibly so.

## The problem

Minimise the discounted total surplus loss over 10 design variables (5 reference years ×
2 pathways) subject to G1–G6. G2–G6 are in `constraints_rte.py`, G1 comes from
`models_optim_complex`:

* **G1** carbon budget
* **G2** blend completeness, χ_B + χ_E ≤ 100
* **G3/G4** biomass and electricity availability
* **G5/G6** ramp-up, the least constraining of a rate and a volume limit (Eq. 12),
  `max{E_{t-1}(1+τ)^Δt, E_{t-1} + ΔE·Δt}`. Both branches are increments on the previous
  period. Every run was redone after the code was found reading the volume branch as an
  absolute ceiling `ΔE·Δt`, which had silently made it a pure rate limit on any mature
  pathway.

## 0. Setup

Only needed for the reading-back sections at the end. The runs themselves need nothing
from this kernel.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optimisation_runs as R

warnings.filterwarnings("ignore")

SWEEP = Path("sweep")
LADDER = SWEEP / "results_e"  # block E
BLOCKS = SWEEP / "results"  # blocks A-D and their references

CASE = "main"  # main | B5 | B75 | B15 | pess

# R.CASES keeps the *published* biomass shares, so the historical runs in ../results stay
# reproducible from it. The 10 % convention is applied at run time by the drivers --
# run_block_e.BIOMASS_OVERRIDE for the ladder, run_batch.BIOMASS_SHARE for blocks A-D --
# rather than edited into the table, so the change is visible instead of silent.
sys.path.insert(0, str(Path("sweep").resolve()))
from run_block_e import BIOMASS_OVERRIDE  # noqa: E402

effective = dict(
    R.CASES[CASE], biomass_share=BIOMASS_OVERRIDE.get(CASE, R.CASES[CASE]["biomass_share"])
)
print(f"{CASE} as published : {R.CASES[CASE]}")
print(f"{CASE} as run here  : {effective}")

## 1. Preflight

Run this first. It fixes the budget every later run is held to, and reports two checks
that decide whether the batch is interpretable.

```sh
cd aeromaps/notebooks/publications/optimisation/migrated/sweep
poetry run python preflight.py
```

Under a minute. It writes `preflight_budget.csv` and
`preflight_refueleu_feasibility.csv`, and prints:

1. **The budget** — ReFuelEU linear, step, and fossil BAU as cumulative GtCO₂ and as a
   world-budget share. It cross-checks the recomputed value against the constant in
   `run_batch.py` and says loudly if they have drifted.
2. **Whether ReFuelEU-linear satisfies G2–G6.** It does not: G5 is violated in 2035 by
   +0.207 — biofuel would have to grow at 25.4 %/yr, to 0.276 EJ, against a cap of
   0.244 EJ set by the volume branch. It *is*
   feasible in four of the nine cap combinations of blocks B/B′ — at 39 %/yr, and at
   0.4 EJ/yr. Under the IEA NZE rate the binding constraint is not biofuel at all but
   **electrofuel in 2050**, violated by 0.96. So the iso-emissions comparison is not
   like-for-like at baseline settings, and that is a finding rather than a blocker.
3. **ε = −1 branch continuity** — 1.03e-07 across −0.999 / −1.0 / −1.001. Passes.
4. **Elasticity-invariance of the 2019-technology reference** — fails, rebound 1.046 at
   ε = −0.6 to 1.101 at ε = −1.4. Not a second anchor: all three consumers read the one
   `initial_airfare_per_rpk`. It is that "2019 technology" is not "2019 prices" — fares
   fall from 0.1218 to 0.0859 against a 0.0924 anchor. `rpk_no_elasticity` *is* exactly
   invariant, so decompositions use that.

## 2. Matched fossil-BAU references

```sh
poetry run python run_references.py
```

Eight MDAs, a couple of minutes. They exist because
`cumulative_total_surplus_loss_discounted` is measured against 2019 unit economics
applied to the exogenous traffic trajectory, **not** against a no-policy scenario. Unit
costs fall over the horizon, so that quantity is negative for every scenario here — fossil
BAU included, at −194 bn €. Only *differences* against a reference are policy costs, and
the reference has to share the run's parameters: the discount rate rescales the whole
series and the elasticity moves the traffic it is summed over.

Eight cover the fourteen runs — five elasticities, the no-feedback formulation, and the
two off-baseline discount rates. Ramp-up caps, biomass share and the electrofuel pathway
need none of their own: fossil BAU burns no alternative fuel. `run_reference_map.csv`
records which reference each run is measured against.

## 3. Blocks A–D — one parameter at a time

```sh
poetry run python run_batch.py --jobs 3      # whole batch, resumable, seeded
poetry run python run_batch.py --list        # the matrix, and what is already done
poetry run python run_batch.py --jobs 2 r_7 r_15   # named runs only
```

Fourteen optimisations, roughly 4 minutes each, so under half an hour at three at a time.

| run | block | varies |
|---|---|---|
| `base` | — | nothing; ε −0.9, 20 %/yr, 0.2 EJ/yr, r 4.5 % |
| `eps_m0_6` `eps_m0_8` `eps_m1_0` `eps_m1_4` | A | price elasticity |
| `fixed_demand` | A′ | no cost feedback at all, `cost` objective |
| `rate_11_8` `rate_39` | B | ramp-up rate cap |
| `vol_0_1` `vol_0_4` | B′ | ramp-up volume cap |
| `efuel_wind` | C | dedicated-wind electrofuel, G4 dropped |
| `r_3_2` `r_7` `r_15` | D | social discount rate |

**Seeding.** With `sweep/previous_optima.csv` present, each run starts from its own
previous optimum, so all fourteen are independent and the whole queue runs `--jobs` at a
time. That is how the corrected re-run was done. With `AEROMAPS_SEED=0`, `base` is a
**barrier** instead: it runs alone, and every other run warm-starts from its optimum.

**Solver settings.** The ftol stop is off; the KKT residual is the intended convergence
criterion, with `max_iter` as the backstop. GEMSEO's xtol stop is still live, and
`summary.csv` records which criterion ended each run — check any that did not end on
KKT. "The objective stopped moving" is not an
optimality test, and warm-starting from the baseline optimum often leaves the objective
flat on the first trial step — at `ftol_abs=1e-8` that stopped `r_3_2` after four
evaluations without it having moved at all. The paper's own default of `1e-3` is looser
still, and shifted mandate shares by up to 11.8 percentage points on the budget ladder.

**`r_3_2` legitimately does not move.** The optimum is a *vertex*: six active G2–G6
constraints, plus the carbon budget, plus three electrofuel variables on their lower
bound — ten active in ten dimensions. The constraints pin the point and the objective only
picks which vertex, so tilting it from 4.5 % to 3.2 % changes nothing. 7 % moves it;
15 % breaks it.

## 4. Block E — the carbon-budget × biomass ladder

```sh
poetry run python run_block_e.py --jobs 3        # all five cases, as continuation chains
poetry run python run_block_e.py --jobs 2 B15 pess
poetry run python run_block_e.py --status

# the corrected re-run: every (case, budget) an independent job, seeded from an earlier
# ladder, detached so it outlives the shell that started it
poetry run python _detach.py logs/blockE.log \
    poetry run python -u run_block_e.py --jobs 4 --seed ../lagrange/results_tight

# section 1 of 02_results: main at the ReFuelEU-equivalent budget, warm-started from
# the 3.2 rung, with a comparison against blocks A-D's `base`
poetry run python run_refueleu_budget.py

# the other cases at the same budget, for the summary figure of 02_results §7.5 --
# independent, so run them together
for c in B5 B75 B15 pess; do
    poetry run python _detach.py logs/refueleu_$c.log \
        poetry run python -u run_refueleu_budget.py $c
done
```

Fifty-five optimisations: ten budgets plus min-CO₂, for each of the five cases, and the
two fixed-mandate references per case. Several hours — B15 is much the slowest, since it
is the only case whose biomass is plentiful enough to reach the blend ceiling.

**Without `--seed`, this is the level at which the problem parallelises.** A case is a continuation chain —
loosest budget first, each run started from the previous optimum — so budgets within a
case are sequential by construction. Cases share nothing and run at once. Nothing below
that is worth the trouble: GEMSEO refuses threads for parallel finite differences
outright (*"all workers shall be different objects"* — the eleven perturbed points share
one MDA object), and processes measured **436 s against 112 s** serial over three SLSQP
iterations, because each point ships the whole 106-discipline chain to a worker and the
workers cannot see the memo `share_mda_across_functions` installs.

Resumable: a run already on disk is skipped and its optimum read back from the HDF.
Delete a file to force it to re-run. **A run that ended infeasible is skipped too** — on
the same start point it would reproduce itself exactly.

Infeasibility at the tight end is a result, not a failure. As it stands: `B5` from 2.4 %,
`B75` and `pess` from 2.2 %, `main` and `B15` from 2.0 %.

## 5. Watching a sweep

```sh
poetry run python status.py                  # blocks A-D, one screen
poetry run python run_block_e.py --status    # block E
watch -n 30 poetry run python status.py

tail -f logs/base.log                        # one run in progress
tail -f logs/blockE_B15.log
```

Every run is a subprocess whose entire output — GEMSEO's logger, the SLSQP progress bar
with its per-iteration objective, any traceback — goes to `logs/<run_id>.log`, in the same
format as the paper's own run logs. `status.py` reads `results/` and `summary.csv` rather
than any log, so it is accurate however the batch was started, and it says so explicitly
when nothing is running.

## 6. Collecting the tables

```sh
poetry run python collect.py
```

Writes three tidy tables from whatever is on disk:

| file | one row per |
|---|---|
| `sweep_by_year.csv` | (run, year) — design variables, fuel shares, RPK, airfare, CO₂, resources, surplus |
| `sweep_summary.csv` | run — parameters, objective, convergence diagnostics, policy cost against its matched reference |
| `sweep_constraints.csv` | (run, constraint, reference year) — value, slack, active, and whether it was in the problem at all |

A constraint dropped from a run — G4 on `efuel_wind` — is reported with `in_problem=False`
rather than omitted, so the table still says what that run *would* have violated.

Then the figures:

```sh
poetry run python plot_elasticity.py       # block A
poetry run python plot_sensitivities.py    # blocks B, B', C, D and the cross-block summary
```

## 7. Reading a ladder back

Everything below is block E, and computes nothing beyond loading JSON. Set `CASE` in the
setup cell.

`min CO₂` is the paper's second problem rather than a budget: the carbon budget
constraint *becomes* the objective and is dropped from the constraint set, leaving G2–G6.
It is the left-hand end of the trade-off curve — the least CO₂ the system can reach at any
cost. Being the tight end, it inherits the last feasible budget's optimum like every other
rung. The published notebooks solved it with NLOPT's MMA; nlopt is not installed here, so
both objectives use SLSQP.

In [ ]:
frames = []
for budget in ["mincarb"] + [
    R.budget_tag(b) for b in [2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8]
]:
    path = LADDER / f"opt_{CASE}_{budget}.json"
    if not path.exists():
        continue
    vector, floats = R.load(path)
    frames.append(
        {
            "budget": budget,
            "CO2 share of world budget (%)": floats["carbon_budget_consumed_share"]
            / R.EU_ASK_SHARE,
            "surplus loss (Bn EUR)": vector["cumulative_total_surplus_loss_discounted"].loc[2050]
            / 1e9,
            "cumulative CO2 (Gt)": vector["cumulative_co2_emissions"].loc[2050],
            "biofuel 2050 (%)": vector["generic_biofuel_share_dropin_fuel"].loc[2050],
            "electrofuel 2050 (%)": vector["generic_electrofuel_share_dropin_fuel"].loc[2050],
            "RPK 2050 (Bn)": vector["rpk"].loc[2050] / 1e9,
        }
    )

pd.DataFrame(frames).round(2)

In [ ]:
# Optimised mandates across the sweep, against the regulation. Compare with Figure 7.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
budgets = [
    b
    for b in [2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8]
    if (LADDER / f"opt_{CASE}_{R.budget_tag(b)}.json").exists()
]
colours = plt.cm.viridis(np.linspace(0.15, 0.9, len(budgets)))

for budget, colour in zip(budgets, colours):
    vector, _ = R.load(LADDER / f"opt_{CASE}_{R.budget_tag(budget)}.json")
    for ax, pathway in zip(axes, ["generic_biofuel", "generic_electrofuel"]):
        ax.plot(
            R.OPTIM_YEARS,
            vector[f"{pathway}_share_dropin_fuel"].loc[R.OPTIM_YEARS],
            "-o",
            ms=3,
            color=colour,
            label=f"{budget}",
        )

for ax, (name, refueleu) in zip(
    axes,
    [
        ("Biofuel", R.REFUELEU_MANDATE["biofuel"]),
        ("Electrofuel", R.REFUELEU_MANDATE["electrofuel"]),
    ],
):
    ax.plot(R.OPTIM_YEARS, refueleu, "--", color="black", lw=1.5, label="ReFuelEU")
    ax.set_title(name)
    ax.set_xlabel("Year")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Drop-in fuel share (%)")
axes[1].legend(title="World budget (%)", fontsize=8, ncol=2, loc="upper left")
fig.suptitle(f"Optimised blending mandate — {R.CASES[CASE]['label']}")
plt.tight_layout()

## 8. Re-running one thing

A single optimisation, outside the continuation — this is how a budget that came out
infeasible is retried, normally with a higher `max_iter` or a different start point:

```python
R.RESULTS_DIR = LADDER
R.run_optimisation(CASE, budget=2.2, max_iter=120,
                   x0={"biofuel": [...], "electrofuel": [...]})
R.run_optimisation(CASE, budget=None)  # min CO2
```

Note that the ladder is a chain: re-running one budget in the middle from a different
start point gives a legitimately different local optimum, and the runs after it in the
chain were started from the old one.

For blocks A–D, delete the run's files and let the driver redo it, so it picks up the
seed and the solver settings:

```sh
rm sweep/results/r_7.* sweep/rows/r_7.json
poetry run python run_batch.py r_7
```

`00_migration_validation.ipynb` section 3 dissects any run from its HDF, and
`02_results.ipynb` draws the paper's figures once everything is on disk.